In [ ]:
import numpy as np
import os
import pandas as pd
import matplotlib.pyplot as plt
import datetime
import scipy
import seaborn as sns
from sqlalchemy.engine import URL
from sqlalchemy import create_engine, text 
import configparser

filepath = os.getcwd()

config = configparser.ConfigParser()
config.read('.secrets.ini')

# Set up SQL Connection
url = URL.create("postgresql",
                 username = config['sql-login']['username'], password=config['sql-login']['password'],
                 database=config['sql-login']['database'], host=config['sql-login']['host']
                 )
dbconn1 = create_engine(url, execution_options={'stream_results':False}).connect()

In [ ]:
# Pull crash reports from the counties affected by the earthquake and sort into Earthquake Days and Control Days
CHPAlerts = pd.read_sql(r'''select *, 
                                case 
                                    when "weekNum" = 0 then 'Earthquake Day'
                                    when "weekNum" < 0 then 'Control Days'
                                    else 'None' 
                                end "EarthquakeControl"
                            from ritis_alerts_mmi_nc75095651 
                            where "weekNum" < 1
                            and "eventType" like '%%Collision%%'
                            and "county" in (
                                'Del Norte',
                                'San Mateo',
                                'San Francisco',
                                'Marin',
                                'Sonoma',
                                'Mendocino',
                                'Humboldt',
                                'Alameda',
                                'Contra Costa',
                                'Solano',
                                'Napa'
                            );''', con=dbconn1, parse_dates=['startTime', 'updateTime'])
WazeAlerts = pd.read_sql(r'''select *,
                                case 
                                    when "weekNum" = 0 then 'Earthquake Day'
                                    when "weekNum" < 0 then 'Control Days'
                                    else 'None' 
                                end "EarthquakeControl"
                          from ritis_alerts_mmi_nc75095651 
                            where "weekNum" < 1
                            and "eventType" like '%%ACCIDENT%%'
                            and "county" in (
                                'Del Norte',
                                'San Mateo',
                                'San Francisco',
                                'Marin',
                                'Sonoma',
                                'Mendocino',
                                'Humboldt',
                                'Alameda',
                                'Contra Costa',
                                'Solano',
                                'Napa'
                            );''', con=dbconn1, parse_dates=['startTime', 'updateTime'])
counties = [c for c in CHPAlerts.groupby('county')['latitude'].min().sort_values(ascending = False).index]
counties.append('All Counties')

In [ ]:
# Manually set times for earthquake and tsunami alert
eqtime = datetime.datetime(year = 2024, month=12, day = 5, hour=13, minute = 44, second = 21, tzinfo=datetime.timezone(-datetime.timedelta(hours=5)))
tsuTime = datetime.datetime(year = 2024, month=12, day = 5, hour=13, minute = 51, second = 15, tzinfo=datetime.timezone(-datetime.timedelta(hours=5)))
tsuTimeCancel = datetime.datetime(year = 2024, month=12, day = 5, hour=14, minute = 54, second = 00, tzinfo=datetime.timezone(-datetime.timedelta(hours=5)))

# Plot alerts to ensure the earthquake time and alert times are correct
CHPAlerts['SecondsFromEarthquake'] = [int((t - eqtime).total_seconds()) % (7*86400) for t in CHPAlerts.startTime]
CHPAlerts['SecondsFromEarthquake'] = [t if t < (7*86400)/2 else t- (7*86400) for t in CHPAlerts.SecondsFromEarthquake]
CHPAlerts['HoursFromEarthquake'] = CHPAlerts.SecondsFromEarthquake/3600.0
ax = sns.scatterplot(data = CHPAlerts[CHPAlerts['weekNum']==-1], x = 'startTime', y='SecondsFromEarthquake')

ax.set_xticklabels(ax.get_xticklabels(), rotation=45)

WazeAlerts['SecondsFromEarthquake'] = [int((t - eqtime).total_seconds()) % (7*86400) for t in WazeAlerts.startTime]
WazeAlerts['SecondsFromEarthquake'] = [t if t < (7*86400)/2 else t- (7*86400) for t in WazeAlerts.SecondsFromEarthquake]
WazeAlerts['HoursFromEarthquake'] = WazeAlerts.SecondsFromEarthquake/3600.0

In [ ]:
# Create violin plot of crash counts in affected counties
fig, axes = plt.subplots(1, 2, sharey=True, sharex=True)
fig.set_figheight(7)
fig.set_figwidth(10)

CHPAlerts_all = CHPAlerts.copy()
CHPAlerts_all['county'] = 'All Counties'
tp = pd.concat([CHPAlerts, CHPAlerts_all], axis = 0)

sns.violinplot(data = tp, ax = axes[0], x = 'HoursFromEarthquake', y='county',
                split=True, hue='EarthquakeControl', inner='stick', order=counties, density_norm='width', cut=0)
sns.lineplot(x=[0, 0+1/3600.0], y=['All Counties', 'Del Norte'], color = 'darkred', label='Earthquake Time', ax = axes[0])
sns.lineplot(x=[((tsuTime-eqtime).total_seconds())/3600.0, ((tsuTime-eqtime).total_seconds() + 1)/3600.0], y=['All Counties', 'Del Norte'],
             color = 'navy', label='Tsunami Warning Time', ax = axes[0])
sns.lineplot(x=[((tsuTimeCancel-eqtime).total_seconds())/3600.0, ((tsuTimeCancel-eqtime).total_seconds() + 1)/3600.0], y=['All Counties', 'Del Norte'],
             color = 'darkgreen', label='Tsunami Warning Cancelled', ax = axes[0])
axes[0].set_xticklabels([(eqtime + pd.Timedelta(hours=xt.get_position()[0]) + pd.Timedelta(hours=-3)).time() for xt in axes[0].get_xticklabels()], rotation=45)
axes[0].get_legend().remove()
axes[0].set_title('CalTrans Crashes')
axes[0].set_xlabel('Local Time of Day (Same Day of Week)')
axes[0].set_ylabel('County Receiving Tsunami Warning')
for ticks in axes[0].collections[1::2]:
    ticks.set_alpha(0.2)


WazeAlerts_all = WazeAlerts.copy()
WazeAlerts_all['county'] = 'All Counties'
tp = pd.concat([WazeAlerts, WazeAlerts_all], axis = 0)

fig.set_figheight(7)
sns.violinplot(data = tp, ax = axes[1], x = 'HoursFromEarthquake', y='county',
                split=True, hue='EarthquakeControl', inner='stick', order=counties, density_norm='width', cut=0)

sns.lineplot(x=[0, 0+1/3600.0], y=['All Counties', 'Del Norte'], color = 'darkred', label='Earthquake Time', ax = axes[1])
sns.lineplot(x=[((tsuTime-eqtime).total_seconds())/3600.0, ((tsuTime-eqtime).total_seconds() + 1)/3600.0], y=['All Counties', 'Del Norte'],
             color = 'navy', label='Tsunami Warning Time', ax = axes[1])
sns.lineplot(x=[((tsuTimeCancel-eqtime).total_seconds())/3600.0, ((tsuTimeCancel-eqtime).total_seconds() + 1)/3600.0], y=['All Counties', 'Del Norte'],
                color = 'darkgreen', label='Tsunami Warning Cancelled', ax = axes[1])
axes[1].set_xticklabels([(eqtime + pd.Timedelta(hours=xt.get_position()[0]) + pd.Timedelta(hours=-3)).time() for xt in axes[0].get_xticklabels()], rotation=45)
axes[1].legend(title='Legend', ncols=3, loc = "lower center", bbox_to_anchor=(-.1, -.3))
axes[1].set_title('Waze Crashes')
axes[1].set_xlabel('Local Time of Day (Same Day of Week)')
axes[1].set_ylabel('County Receiving Tsunami Warning')
for ticks in axes[1].collections[1::2]:
    ticks.set_alpha(0.2)

plt.savefig('Fig14.eps', bbox_inches = 'tight', pad_inches = 0.5)
plt.show()

In [ ]:
# Determine the statistical significance of Mendocino results
fig, ax = plt.subplots(1, 2, sharex=True, sharey=True)

fig.set_figheight(6)
fig.set_figwidth(10)

for offset in np.arange(-2, 2, 0.5): 
    # Police-reported alerts   
    test = CHPAlerts[(CHPAlerts['HoursFromEarthquake']>offset)&
                    (CHPAlerts['HoursFromEarthquake']<offset+1)]
    CF = []
    EF = []
    M = []
    for m in np.arange(2, 8, 0.25):

        print("---------------------")
        print(f"**** poisson means test results for mmi at least {m} ****")
        kc = test[(test['EarthquakeControl']=='Control Days')&(test['MMI_Value']>=m)].uuid.count()
        nc = test[(test['EarthquakeControl']=='Control Days')].uuid.count()
        ka = test[(test['EarthquakeControl']=='Earthquake Day')&(test['MMI_Value']>=m)].uuid.count()
        na = test[(test['EarthquakeControl']=='Earthquake Day')].uuid.count()
        print(f'{kc}/{nc} = {kc/nc}, {ka}/{na}={ka/na}')

        CF.append(kc/nc)
        EF.append(ka/na)
        M.append(m)
        print(scipy.stats.poisson_means_test(kc, nc, ka, na, alternative='two-sided'))
        print("---------------------")

    sns.lineplot(x=M, y=[e-c for e, c in zip(EF, CF)], ax = ax[0], label = f"{offset} hours to {offset+1} hours")

    # Crowdsourced alerts
    test = WazeAlerts[(WazeAlerts['HoursFromEarthquake']>offset)&
                    (WazeAlerts['HoursFromEarthquake']<offset+1)]
    CF = []
    EF = []
    M = []
    for m in np.arange(2, 8, 0.25):

        print("---------------------")
        print(f"**** poisson means test results for mmi at least {m} ****")
        kc = test[(test['EarthquakeControl']=='Control Days')&(test['MMI_Value']>=m)].uuid.count()
        nc = test[(test['EarthquakeControl']=='Control Days')].uuid.count()
        ka = test[(test['EarthquakeControl']=='Earthquake Day')&(test['MMI_Value']>=m)].uuid.count()
        na = test[(test['EarthquakeControl']=='Earthquake Day')].uuid.count()
        print(f'{kc}/{nc} = {kc/nc}, {ka}/{na}={ka/na}')

        CF.append(kc/nc)
        EF.append(ka/na)
        M.append(m)
        print(scipy.stats.poisson_means_test(kc, nc, ka, na, alternative='two-sided'))
        print("---------------------")

    # Plot changes in crash rates
    sns.lineplot(x=M, y=[e-c for e, c in zip(EF, CF)], ax = ax[1], label = f"{offset} hours to {offset+1} hours")

ax[0].set_xlabel("MMI Threshold")
ax[0].set_ylabel("Difference in Earthquake to Control Crash Fractions")
ax[0].set_title("CalTrans Crashes")
ax[1].set_xlabel("MMI Threshold")
ax[1].set_ylabel("Difference in Earthquake to Control Crash Fractions")
ax[1].set_title("Waze Crashes")
ax[0].get_legend().remove()

for axis in ax:
    colormap = plt.cm.viridis   
    colors = [colormap(i) for i in np.linspace(0, 1, len(axis.lines))]
    for i, j in enumerate(axis.lines):
        j.set_color(colors[i])

ax[1].legend(title='Legend', ncols=4, loc = "lower center", bbox_to_anchor=(-.1, -.3))